# Bag of Words and TF-IDF

Machine learning models require numerical input. The first major breakthrough in statistical NLP was representing documents as **vectors of word counts** — treating each document as an unordered collection of words, a "bag" from which order has been discarded. This simple idea powered spam filters, search engines, and document classifiers for decades.

**TF-IDF** (Term Frequency–Inverse Document Frequency) refines raw counts by downweighting words that appear everywhere and upweighting words that are distinctive to a specific document. Together, bag-of-words and TF-IDF form the foundation of classical text feature engineering — and remain relevant as baselines, interpretable features, and building blocks for hybrid systems.

This notebook builds both representations from scratch, implements them with scikit-learn, and demonstrates their use in document classification and similarity search.

In [ ]:
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from sklearn.naive_bayes import MultinomialNB

np.set_printoptions(precision=4, suppress=True)
plt.style.use("seaborn-v0_8-whitegrid")
np.random.seed(42)

---

## 1. From Text to Numbers

A document like *"the cat sat on the mat"* must become a fixed-length numeric vector before a classifier can process it. The **bag-of-words (BoW)** model does this by:

1. Building a **vocabulary** of all unique words across the corpus.
2. For each document, counting how many times each vocabulary word appears.
3. Representing the document as a vector of those counts.

```
Vocabulary: [the, cat, sat, on, mat, dog, ran]

Doc 1: "the cat sat on the mat"  →  [2, 1, 1, 1, 1, 0, 0]
Doc 2: "the dog ran"              →  [1, 0, 0, 0, 0, 1, 1]
```

The name "bag of words" reflects that word **order is discarded**. *"The dog bit the man"* and *"The man bit the dog"* produce identical vectors — a fundamental limitation we address later with n-grams and neural models.

---

## 2. The Document-Term Matrix

When every document in a corpus is vectorized, the result is a **document-term matrix** $\mathbf{X}$ of shape $(N \times V)$:

- $N$ = number of documents
- $V$ = vocabulary size
- $X_{ij}$ = count of word $j$ in document $i$

Each **row** is a document; each **column** is a word. The matrix is typically **sparse** — most entries are zero because any single document uses only a small fraction of the full vocabulary.

In [ ]:
corpus = [
    "the cat sat on the mat",
    "the dog ran in the park",
    "the cat and dog are friends",
]

def tokenize(text):
    return re.findall(r"\b[a-z]+\b", text.lower())

def build_vocabulary(documents):
    vocab = sorted(set(w for doc in documents for w in tokenize(doc)))
    return {word: idx for idx, word in enumerate(vocab)}

def bow_vector(text, vocab):
    counts = Counter(tokenize(text))
    return np.array([counts.get(word, 0) for word in vocab])

vocab_list = sorted(build_vocabulary(corpus).keys())
vocab_idx = {w: i for i, w in enumerate(vocab_list)}

matrix = np.array([bow_vector(doc, vocab_list) for doc in corpus])

df = pd.DataFrame(matrix, columns=vocab_list,
                  index=[f"Doc {i+1}" for i in range(len(corpus))])
print("Document-Term Matrix (word counts):")
print(df.to_string())

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3))
im = ax.imshow(matrix, cmap="YlOrRd", aspect="auto")
ax.set_xticks(range(len(vocab_list)))
ax.set_xticklabels(vocab_list, rotation=45, ha="right")
ax.set_yticks(range(len(corpus)))
ax.set_yticklabels([f"Doc {i+1}" for i in range(len(corpus))])
ax.set_title("Document-Term Matrix Heatmap")
plt.colorbar(im, label="Count")
plt.tight_layout()
plt.show()

---

## 3. Binary Bag of Words

Instead of counts, **binary BoW** records only whether a word is present (1) or absent (0). This prevents long documents from dominating simply because they contain more words.

$$X_{ij} = \begin{cases} 1 & \text{if word } j \text{ appears in document } i \\ 0 & \text{otherwise} \end{cases}$$

Binary BoW is common in topic modeling and when word presence matters more than frequency.

In [ ]:
binary_matrix = (matrix > 0).astype(int)
df_binary = pd.DataFrame(binary_matrix, columns=vocab_list,
                          index=[f"Doc {i+1}" for i in range(len(corpus))])
print("Binary Bag of Words:")
print(df_binary.to_string())

---

## 4. N-gram Bag of Words

To capture limited word order, BoW can be extended to **n-grams** — contiguous sequences of $n$ words treated as single tokens.

- **Unigrams:** `cat`, `sat`, `mat`
- **Bigrams:** `the cat`, `cat sat`, `sat on`
- **Trigrams:** `the cat sat`, `cat sat on`

The phrase *"not good"* as a bigram carries different sentiment than the unigrams *"not"* and *"good"* alone. Combining unigrams and bigrams is a common practice:

```python
CountVectorizer(ngram_range=(1, 2))  # unigrams + bigrams
```

The tradeoff: vocabulary size grows rapidly. With $V$ unigrams, there can be up to $V^2$ bigrams.

In [ ]:
def generate_ngrams(tokens, n):
    return [" ".join(tokens[i:i+n]) for i in range(len(tokens) - n + 1)]

text = "the cat sat on the mat"
tokens = tokenize(text)

for n in [1, 2, 3]:
    ngrams = generate_ngrams(tokens, n)
    print(f"{n}-grams: {ngrams}")

In [ ]:
bigram_vectorizer = CountVectorizer(ngram_range=(2, 2))
bigram_matrix = bigram_vectorizer.fit_transform(corpus).toarray()
bigram_features = bigram_vectorizer.get_feature_names_out()

print(f"Number of bigram features: {len(bigram_features)}")
print(f"Bigram vocabulary: {list(bigram_features)}")
print(f"\nDoc 1 bigram vector (first 8): {bigram_matrix[0, :8]}")

---

## 5. scikit-learn CountVectorizer

scikit-learn's `CountVectorizer` handles tokenization, vocabulary building, and matrix construction in one step. Key parameters:

| Parameter | Purpose |
|-----------|--------|
| `lowercase` | Convert to lowercase (default True) |
| `stop_words` | Remove common words (`'english'` or custom list) |
| `ngram_range` | Tuple `(min_n, max_n)` for n-gram range |
| `max_features` | Keep only top N most frequent terms |
| `min_df` | Ignore terms appearing in fewer than N documents |
| `max_df` | Ignore terms appearing in more than N documents (or fraction) |
| `binary` | If True, record presence only |

In [ ]:
vectorizer = CountVectorizer(stop_words="english")
X_counts = vectorizer.fit_transform(corpus)

print(f"Matrix shape: {X_counts.shape}  (documents × vocabulary)")
print(f"Vocabulary: {vectorizer.get_feature_names_out()}")
print(f"\nSparse representation (only non-zero entries stored):")
print(X_counts)

---

## 6. Limitations of Raw Counts

Raw word counts have two major problems:

**Problem 1: Common words dominate.** Words like *"the"*, *"is"*, and *"of"* appear frequently in every document. They have high counts but carry little discriminative information about document topic or class.

**Problem 2: Long documents have larger counts.** A 10-page report naturally has higher word counts than a one-paragraph summary, even if both discuss the same topic. Raw counts bias toward document length rather than content.

**TF-IDF** addresses both problems by weighting each word based on how important it is to a specific document relative to the entire corpus.

---

## 7. Term Frequency (TF)

**Term frequency** measures how often a term appears in a document. Several variants exist:

**Raw count:**

$$\text{tf}(t, d) = \text{count of } t \text{ in } d$$

**Normalized by document length:**

$$\text{tf}(t, d) = \frac{\text{count of } t \text{ in } d}{\text{total words in } d}$$

**Log normalization** (sublinear scaling — dampens the effect of very high frequencies):

$$\text{tf}(t, d) = 1 + \log(\text{count of } t \text{ in } d)$$

Log normalization prevents a word appearing 100 times from having 100× the influence of a word appearing once.

---

## 8. Inverse Document Frequency (IDF)

**Inverse document frequency** measures how rare a term is across the corpus. Words that appear in many documents are less informative.

$$\text{idf}(t) = \log\frac{N}{|\{d : t \in d\}|}$$

where $N$ is the total number of documents and the denominator is the number of documents containing term $t$.

A word appearing in every document gets $\text{idf} = \log(1) = 0$ — it is completely downweighted. A word appearing in only one document gets $\text{idf} = \log(N)$ — maximum weight.

**Smooth IDF** (used by scikit-learn) adds 1 to avoid division by zero and to give unseen terms a non-zero weight:

$$\text{idf}(t) = \log\frac{N + 1}{|\{d : t \in d\}| + 1} + 1$$

---

## 9. TF-IDF Combined

The TF-IDF score for term $t$ in document $d$ is:

$$\text{tf-idf}(t, d) = \text{tf}(t, d) \times \text{idf}(t)$$

High TF-IDF means: the word appears frequently in this document (high TF) but rarely in other documents (high IDF). These are the **discriminative** words that characterize a document.

For our three-document corpus:
- *"cat"* appears in 2 of 3 documents → moderate IDF.
- *"park"* appears in only 1 document → high IDF — a strong identifier for Doc 2.
- *"the"* appears in all documents → IDF near zero.

In [ ]:
def compute_tfidf(documents):
    """Compute TF-IDF matrix from scratch."""
    vocab = build_vocabulary(documents)
    vocab_list = sorted(vocab.keys())
    N = len(documents)

    # Term frequency matrix
    tf = np.array([bow_vector(doc, vocab_list) for doc in documents], dtype=float)

    # Document frequency: how many docs contain each term
    df = (tf > 0).sum(axis=0)

    # IDF with smoothing
    idf = np.log((N + 1) / (df + 1)) + 1

    # TF-IDF
    tfidf = tf * idf
    return tfidf, vocab_list, idf

tfidf_matrix, vocab_list, idf_scores = compute_tfidf(corpus)

df_tfidf = pd.DataFrame(
    np.round(tfidf_matrix, 3), columns=vocab_list,
    index=[f"Doc {i+1}" for i in range(len(corpus))]
)
print("TF-IDF Matrix:")
print(df_tfidf.to_string())
print(f"\nIDF scores (higher = more distinctive):")
for word, score in sorted(zip(vocab_list, idf_scores), key=lambda x: -x[1]):
    print(f"  {word:10s} {score:.3f}")

---

## 10. scikit-learn TfidfVectorizer

`TfidfVectorizer` combines `CountVectorizer` with TF-IDF weighting. It accepts the same parameters plus:

| Parameter | Purpose |
|-----------|--------|
| `sublinear_tf` | Use $1 + \log(\text{tf})$ instead of raw tf |
| `norm` | Normalize vectors (`'l2'` default, `'l1'`, or `None`) |
| `use_idf` | Enable/disable IDF weighting |
| `smooth_idf` | Add 1 to document frequencies (default True) |

In [ ]:
tfidf_vectorizer = TfidfVectorizer(stop_words="english")
X_tfidf = tfidf_vectorizer.fit_transform(corpus)

print(f"TF-IDF matrix shape: {X_tfidf.shape}")
print(f"Features: {tfidf_vectorizer.get_feature_names_out()}")

df_sk = pd.DataFrame(
    np.round(X_tfidf.toarray(), 3),
    columns=tfidf_vectorizer.get_feature_names_out(),
    index=[f"Doc {i+1}" for i in range(len(corpus))]
)
print("\nscikit-learn TF-IDF:")
print(df_sk.to_string())

---

## 11. L2 Normalization

By default, `TfidfVectorizer` applies **L2 normalization** to each document vector, scaling it to unit length:

$$\mathbf{v}_{\text{norm}} = \frac{\mathbf{v}}{\|\mathbf{v}\|_2}$$

This makes cosine similarity equivalent to the dot product — simplifying document comparison. Without normalization, longer documents would have larger TF-IDF vectors regardless of content relevance.

In [ ]:
tfidf_raw = TfidfVectorizer(stop_words="english", norm=None)
tfidf_l2 = TfidfVectorizer(stop_words="english", norm="l2")

raw_vecs = tfidf_raw.fit_transform(corpus).toarray()
l2_vecs = tfidf_l2.fit_transform(corpus).toarray()

print("L2 norms of document vectors:")
print(f"  Without normalization: {np.linalg.norm(raw_vecs, axis=1)}")
print(f"  With L2 normalization:  {np.linalg.norm(l2_vecs, axis=1)}")

---

## 12. Cosine Similarity and Document Search

TF-IDF vectors enable **document similarity search**. The **cosine similarity** between two vectors measures the angle between them (ignoring magnitude):

$$\text{sim}(\mathbf{a}, \mathbf{b}) = \frac{\mathbf{a} \cdot \mathbf{b}}{\|\mathbf{a}\| \|\mathbf{b}\|}$$

With L2-normalized TF-IDF vectors, this simplifies to the dot product. A query document is vectorized, compared against all corpus documents, and the most similar are returned — the core mechanism behind keyword search and information retrieval.

In [ ]:
search_corpus = [
    "python machine learning tutorial for beginners",
    "deep learning neural networks with pytorch",
    "best restaurants in new york city",
    "machine learning algorithms for text classification",
    "italian food recipes and cooking tips",
    "natural language processing with sklearn",
]

search_vec = TfidfVectorizer(stop_words="english")
doc_matrix = search_vec.fit_transform(search_corpus)

query = "machine learning text classification"
query_vec = search_vec.transform([query])

# Cosine similarity (dot product of L2-normalized vectors)
similarities = (doc_matrix @ query_vec.T).toarray().flatten()
ranked = np.argsort(similarities)[::-1]

print(f"Query: '{query}'\n")
print(f"{'Rank':<5s} {'Score':>6s}  Document")
print("-" * 60)
for rank, idx in enumerate(ranked, 1):
    print(f"{rank:<5d} {similarities[idx]:6.3f}  {search_corpus[idx]}")

---

## 13. Text Classification with BoW and TF-IDF

The standard classical NLP pipeline: vectorize text, then train a classifier. **Logistic regression** and **naive Bayes** work particularly well with sparse text features.

In [ ]:
# Sentiment classification dataset
reviews = [
    ("this movie is fantastic and amazing", "positive"),
    ("wonderful acting and great storyline", "positive"),
    ("best film I have ever seen", "positive"),
    ("brilliant performance by the lead actor", "positive"),
    ("absolutely loved every minute of it", "positive"),
    ("terrible movie waste of time", "negative"),
    ("awful acting and boring plot", "negative"),
    ("worst film ever made", "negative"),
    ("horrible experience do not watch", "negative"),
    ("completely disappointed and frustrated", "negative"),
    ("decent film with good moments", "positive"),
    ("not great but not terrible either", "negative"),
    ("pretty good overall enjoyable watch", "positive"),
    ("mediocre at best nothing special", "negative"),
    ("excellent cinematography and direction", "positive"),
    ("poor script and weak characters", "negative"),
    ("outstanding masterpiece of cinema", "positive"),
    ("dreadful and painful to sit through", "negative"),
    ("really enjoyed the film thoroughly", "positive"),
    ("boring and predictable storyline", "negative"),
]

texts = [r[0] for r in reviews]
labels = [r[1] for r in reviews]
X_train, X_test, y_train, y_test = train_test_split(
    texts, labels, test_size=0.3, random_state=42, stratify=labels
)

configs = {
    "BoW (counts)": CountVectorizer(stop_words="english"),
    "TF-IDF": TfidfVectorizer(stop_words="english"),
    "TF-IDF + bigrams": TfidfVectorizer(stop_words="english", ngram_range=(1, 2)),
}

print(f"{'Representation':<22s} {'Accuracy':>8s}")
print("-" * 32)
for name, vec in configs.items():
    X_tr = vec.fit_transform(X_train)
    X_te = vec.transform(X_test)
    clf = LogisticRegression(max_iter=1000, random_state=42)
    clf.fit(X_tr, y_train)
    acc = accuracy_score(y_test, clf.predict(X_te))
    print(f"{name:<22s} {acc:8.3f}")

In [ ]:
# Most important features for sentiment (TF-IDF + logistic regression)
vec = TfidfVectorizer(stop_words="english", ngram_range=(1, 2))
X_tr = vec.fit_transform(X_train)
clf = LogisticRegression(max_iter=1000, random_state=42)
clf.fit(X_tr, y_train)

features = vec.get_feature_names_out()
coefs = clf.coef_[0]
top_pos = np.argsort(coefs)[-8:][::-1]
top_neg = np.argsort(coefs)[:8]

print("Top positive indicators:")
for i in top_pos:
    print(f"  {features[i]:25s} {coefs[i]:+.3f}")
print("\nTop negative indicators:")
for i in top_neg:
    print(f"  {features[i]:25s} {coefs[i]:+.3f}")

---

## 14. Controlling Vocabulary Size

Large corpora produce vocabularies of hundreds of thousands of words — most appearing only once (typos, rare terms). Controlling vocabulary size reduces memory, speeds training, and reduces overfitting.

| Parameter | Effect |
|-----------|--------|
| `max_features=5000` | Keep only the 5000 most frequent terms |
| `min_df=2` | Ignore terms appearing in fewer than 2 documents |
| `max_df=0.95` | Ignore terms appearing in more than 95% of documents |

`min_df` removes rare noise. `max_df` removes near-universal words that IDF would downweight anyway but that still consume memory.

In [ ]:
large_reviews = texts * 5  # simulate larger corpus

vocab_configs = [
    ("No filtering", {}),
    ("min_df=2", {"min_df": 2}),
    ("max_df=0.8", {"max_df": 0.8}),
    ("max_features=20", {"max_features": 20}),
]

print(f"{'Config':<20s} {'Vocab Size':>10s}")
print("-" * 32)
for name, params in vocab_configs:
    v = TfidfVectorizer(stop_words="english", **params)
    v.fit(large_reviews)
    print(f"{name:<20s} {len(v.vocabulary_):10d}")

---

## 15. Naive Bayes for Text

**Multinomial Naive Bayes** is a classic text classifier that models word counts with a probabilistic framework. It assumes word features are conditionally independent given the class — the "naive" assumption — which is unrealistic but works surprisingly well for text.

$$P(c \mid \mathbf{d}) \propto P(c) \prod_{i} P(w_i \mid c)$$

Naive Bayes is fast, handles high-dimensional sparse data well, and needs little training data. It pairs naturally with count vectors.

In [ ]:
bow_vec = CountVectorizer(stop_words="english")
X_tr_bow = bow_vec.fit_transform(X_train)
X_te_bow = bow_vec.transform(X_test)

nb = MultinomialNB()
nb.fit(X_tr_bow, y_train)
nb_acc = accuracy_score(y_test, nb.predict(X_te_bow))

lr_vec = TfidfVectorizer(stop_words="english")
X_tr_tfidf = lr_vec.fit_transform(X_train)
X_te_tfidf = lr_vec.transform(X_test)
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_tr_tfidf, y_train)
lr_acc = accuracy_score(y_test, lr.predict(X_te_tfidf))

print(f"Multinomial Naive Bayes + BoW:  {nb_acc:.3f}")
print(f"Logistic Regression + TF-IDF:   {lr_acc:.3f}")

---

## 16. Bag of Words vs TF-IDF

| Aspect | Bag of Words | TF-IDF |
|--------|-------------|--------|
| **Values** | Raw counts (or binary) | Weighted by term importance |
| **Common words** | High counts dominate | Downweighted by IDF |
| **Long documents** | Higher counts | Normalized (especially with L2) |
| **Best for** | Naive Bayes, topic modeling (LDA) | Logistic regression, SVM, search |
| **Interpretability** | Direct word counts | Weighted importance scores |

Neither captures word order (beyond n-grams), semantic similarity (*"car"* and *"automobile"* are different features), or context. Dense word embeddings and transformers address these limitations — but BoW and TF-IDF remain strong, fast baselines.

---

## 17. Sparsity and Memory

A vocabulary of 50,000 words and 10,000 documents produces a $10{,}000 \times 50{,}000$ matrix with 500 million potential entries — but typically fewer than 1% are non-zero. Storing this as a dense array would waste enormous memory.

scikit-learn uses **sparse matrices** (CSR format) that store only non-zero values and their positions. A sparse TF-IDF matrix with 1% density uses roughly 100× less memory than its dense equivalent. Always keep text feature matrices sparse until you explicitly need dense arrays.

In [ ]:
X_sparse = TfidfVectorizer(stop_words="english", max_features=1000).fit_transform(texts * 10)
density = X_sparse.nnz / (X_sparse.shape[0] * X_sparse.shape[1])

print(f"Matrix shape:     {X_sparse.shape}")
print(f"Non-zero entries: {X_sparse.nnz:,}")
print(f"Total entries:    {X_sparse.shape[0] * X_sparse.shape[1]:,}")
print(f"Density:          {density:.4f} ({density*100:.2f}%)")
print(f"Memory (sparse):  {X_sparse.data.nbytes / 1024:.1f} KB")
print(f"Memory (dense):   {X_sparse.toarray().nbytes / 1024:.1f} KB")

---

## 18. Best Practices

**Always fit on training data only.** Call `fit_transform` on training set and `transform` on validation/test. Fitting on the full dataset leaks vocabulary and IDF statistics.

**Start with TF-IDF + logistic regression.** Strong, fast baseline for most text classification tasks.

**Try bigrams.** `ngram_range=(1, 2)` often improves performance with modest vocabulary growth.

**Tune `min_df` and `max_df`.** Remove rare noise and near-universal terms before they bloat the vocabulary.

**Do not convert to dense unless necessary.** Neural networks may need dense input, but classical ML works directly on sparse matrices.

**Compare against a baseline.** Even a simple BoW model establishes the floor that more complex approaches must beat.

---

## 19. Summary

**Bag of words** represents documents as vectors of word counts, discarding word order. The **document-term matrix** rows are documents and columns are vocabulary words — typically very sparse.

**TF-IDF** weights each word by its frequency in the document (TF) and its rarity across the corpus (IDF). Common words are downweighted; distinctive words are highlighted. L2 normalization enables efficient cosine similarity search.

scikit-learn's `CountVectorizer` and `TfidfVectorizer` handle tokenization, vocabulary filtering, and matrix construction. Combined with logistic regression or naive Bayes, they form a powerful classical NLP pipeline.

While modern neural models have surpassed these methods on many benchmarks, bag-of-words and TF-IDF remain essential tools — fast, interpretable, and effective baselines for text classification, search, and feature engineering.